# JMA Typhoon Encounter Analysis (1979-2025)

This notebook performs a spatial query on the JMA (Japan Meteorological Agency) Best Track dataset to identify typhoons that passed within **150 km** of a specific target coordinate.

### Target Coordinate:
- **Longitude**: E118.923708
- **Latitude**: N23.990464

### Key Features of this Analysis:
1. **Date Window Filtering**: Supports custom date range queries down to the monthly resolution (e.g. `1979-01` to `2025-12`).
2. **Data Extraction & Unit Conversion**:
   - Converts wind speed from knots (kt) to meters per second (m/s).
   - Converts wind radii from nautical miles (nm) to kilometers (km).
   - Removes JMA-specific internal columns (`Indicator`, `Grade`, `Landfall`).
   - Adds `Elapsed_Hours`, `Ambient_Pressure_hPa` (customizable), and `Distance_to_Target_km`.
   - Saves each typhoon record individually as `{Year}_{Name}.xlsx` in the `results/` directory.
3. **Yearly Path Visualization**: Generates a high-quality map for each year featuring all encounter typhoons with storm paths, marker sizes scaled by pressure deficit (`Background_Pressure - Center_Pressure`), high-resolution mapping layers, and a 150 km boundary circle.
4. **Master Map**: Produces a wide-region overview map showing all encounter typhoon paths across history.
5. **Summary Reporting**: Generates a compiled summary list of all encounter events saved as an Excel spreadsheet.

## 1. Environment Setup and Library Imports

In [ ]:
import os
import math
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("Libraries successfully imported!")

## 2. Configuration and Constants

In [ ]:
# Target coordinates and search radius
TARGET_LON = 118.923708
TARGET_LAT = 23.990464
THRESHOLD_KM = 150.0

# Date range filter
START_DATE_STR = "1979-01"
END_DATE_STR = "2025-12"

# Constant Background ambient pressure
BACKGROUND_PRESSURE = 1013.0

# Map bounds
MAP_MIN_LON, MAP_MAX_LON = 95.0, 160.0
MAP_MIN_LAT, MAP_MAX_LAT = 0.0, 50.0

# File paths
BST_FILE_PATH = r"../Typhoon/bst_all.txt"
if not os.path.exists(BST_FILE_PATH):
    # Fallback to absolute path if relative is not resolved
    BST_FILE_PATH = r"c:\AntigravityZone\HT\Typhoon\bst_all.txt"

RESULTS_DIR = r"./results"
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

print(f"Target Coordinates: E{TARGET_LON}, N{TARGET_LAT}")
print(f"Date Range: {START_DATE_STR} to {END_DATE_STR}")
print(f"Background Pressure: {BACKGROUND_PRESSURE} hPa")
print(f"Output Directory: {RESULTS_DIR}")

## 3. Utility Functions

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """Calculate the great circle distance between two points in kilometers."""
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return c * 6371.0

def geodesic_circle(center_lon, center_lat, radius_km, num_points=200):
    """Generate points describing a circle on a sphere at a target point."""
    R = 6371.0
    r_rad = radius_km / R
    lat_rad = np.radians(center_lat)
    lon_rad = np.radians(center_lon)
    angles = np.linspace(0, 2*np.pi, num_points)
    
    circle_lats = np.arcsin(np.sin(lat_rad)*np.cos(r_rad) + np.cos(lat_rad)*np.sin(r_rad)*np.cos(angles))
    circle_lons = lon_rad + np.arctan2(np.sin(angles)*np.sin(r_rad)*np.cos(lat_rad), 
                                      np.cos(r_rad) - np.sin(lat_rad)*np.sin(circle_lats))
    return np.degrees(circle_lons), np.degrees(circle_lats)

def parse_time(time_str):
    """Parse yymmddhh JMA time format into a datetime object."""
    yy = int(time_str[0:2])
    mm = int(time_str[2:4])
    dd = int(time_str[4:6])
    hh = int(time_str[6:8])
    year = 1900 + yy if yy >= 50 else 2000 + yy
    return datetime.datetime(year, mm, dd, hh)

def draw_target(ax, target_lon, target_lat, threshold, pin_img, width_deg):
    """Draw target pin image overlay and range boundary circle."""
    if pin_img is not None:
        pin_w = max(0.08, width_deg * 0.045)
        extent = [target_lon - pin_w/2, target_lon + pin_w/2, target_lat, target_lat + pin_w]
        ax.imshow(pin_img, extent=extent, transform=ccrs.PlateCarree(), zorder=10, label='Target Point')
    else:
        ax.plot(target_lon, target_lat, marker='o', color='red', markersize=10, label='Target Point', zorder=10)
        
    circle_lons, circle_lats = geodesic_circle(target_lon, target_lat, threshold)
    ax.plot(circle_lons, circle_lats, 'r--', linewidth=1.5, label=f'{threshold} km Boundary', zorder=9)

## 4. JMA Best Track Data Parser

In [ ]:
start_date = datetime.datetime.strptime(START_DATE_STR, "%Y-%m")
end_date = datetime.datetime.strptime(END_DATE_STR, "%Y-%m")

print("Parsing best tracks file...")
with open(BST_FILE_PATH, 'r') as f:
    lines = f.readlines()

i = 0
encounters = []
current_header = None
current_data = []

while i < len(lines):
    line = lines[i]
    if line.startswith('66666'):
        if current_header and current_data:
            is_encounter = False
            min_dist = float('inf')
            min_dist_idx = -1
            for idx, dp in enumerate(current_data):
                dt = parse_time(dp['time_str'])
                if start_date <= dt <= end_date:
                    dist = haversine(dp['lon'], dp['lat'], TARGET_LON, TARGET_LAT)
                    dp['dist_to_target'] = dist
                    if dist < min_dist:
                        min_dist = dist
                        min_dist_idx = idx
                    if dist < THRESHOLD_KM:
                        is_encounter = True
            
            if is_encounter:
                encounters.append({
                    'header': current_header,
                    'data': current_data,
                    'min_dist': min_dist,
                    'min_dist_idx': min_dist_idx
                })
        
        raw = line.rstrip('\r\n')
        tc_id = raw[6:10].strip()
        data_lines_count = int(raw[12:15].strip())
        name = raw[30:50].strip() or "UNNAMED"
        
        yy_val = int(tc_id[:2])
        year = 1900 + yy_val if yy_val >= 50 else 2000 + yy_val
        
        current_header = {
            'tc_id': tc_id,
            'data_lines_count': data_lines_count,
            'name': name,
            'year': year
        }
        current_data = []
        i += 1
    else:
        raw = line.rstrip('\r\n')
        time_ana = raw[0:8].strip()
        indicator = raw[9:12].strip()
        grade = raw[13:14].strip()
        lat_raw = float(raw[15:18].strip()) / 10.0
        lon_raw = float(raw[19:23].strip()) / 10.0
        press = float(raw[24:28].strip())
        
        max_wind = raw[33:36].strip()
        max_wind = float(max_wind) if max_wind else None
        
        dir_50 = raw[41:42].strip()
        rad_50_l = raw[42:46].strip()
        rad_50_l = float(rad_50_l) if (rad_50_l and rad_50_l != '0000') else None
        rad_50_s = raw[47:51].strip()
        rad_50_s = float(rad_50_s) if (rad_50_s and rad_50_s != '0000') else None
        
        dir_30 = raw[52:53].strip()
        rad_30_l = raw[53:57].strip()
        rad_30_l = float(rad_30_l) if (rad_30_l and rad_30_l != '0000') else None
        rad_30_s = raw[58:62].strip()
        rad_30_s = float(rad_30_s) if (rad_30_s and rad_30_s != '0000') else None
        
        landfall = raw[71:72].strip()
        
        current_data.append({
            'time_str': time_ana,
            'indicator': indicator,
            'grade': grade,
            'lat': lat_raw,
            'lon': lon_raw,
            'press': press,
            'max_wind': max_wind,
            'dir_50': dir_50,
            'rad_50_l': rad_50_l,
            'rad_50_s': rad_50_s,
            'dir_30': dir_30,
            'rad_30_l': rad_30_l,
            'rad_30_s': rad_30_s,
            'landfall': landfall
        })
        i += 1

if current_header and current_data:
    is_encounter = False
    min_dist = float('inf')
    min_dist_idx = -1
    for idx, dp in enumerate(current_data):
        dt = parse_time(dp['time_str'])
        if start_date <= dt <= end_date:
            dist = haversine(dp['lon'], dp['lat'], TARGET_LON, TARGET_LAT)
            dp['dist_to_target'] = dist
            if dist < min_dist:
                min_dist = dist
                min_dist_idx = idx
            if dist < THRESHOLD_KM:
                is_encounter = True
    if is_encounter:
        encounters.append({
            'header': current_header,
            'data': current_data,
            'min_dist': min_dist,
            'min_dist_idx': min_dist_idx
        })

print(f"Parsed finished. Total encounter storms: {len(encounters)}")

## 5. Data Cleansing, Unit Conversion & Excel Export

In [ ]:
summary_list = []

print("Processing encounters and exporting spreadsheets...")
for idx, enc in enumerate(encounters):
    h = enc['header']
    d_list = enc['data']
    t0 = parse_time(d_list[0]['time_str'])
    
    parsed_rows = []
    for dp in d_list:
        dt = parse_time(dp['time_str'])
        elapsed = (dt - t0).total_seconds() / 3600.0
        
        max_wind_mps = dp['max_wind'] * 0.51444 if dp['max_wind'] is not None else None
        rad_50_l_km = dp['rad_50_l'] * 1.852 if dp['rad_50_l'] is not None else None
        rad_50_s_km = dp['rad_50_s'] * 1.852 if dp['rad_50_s'] is not None else None
        rad_30_l_km = dp['rad_30_l'] * 1.852 if dp['rad_30_l'] is not None else None
        rad_30_s_km = dp['rad_30_s'] * 1.852 if dp['rad_30_s'] is not None else None
        
        parsed_rows.append({
            'Time': dt.strftime('%Y-%m-%d %H:00:00'),
            'Elapsed_Hours': elapsed,
            'Ambient_Pressure_hPa': BACKGROUND_PRESSURE,
            'Distance_to_Target_km': round(dp['dist_to_target'], 2),
            'Latitude_deg': dp['lat'],
            'Longitude_deg': dp['lon'],
            'Central_Pressure_hPa': dp['press'],
            'Max_Wind_Speed_mps': round(max_wind_mps, 2) if max_wind_mps is not None else None,
            '50kt_Wind_Longest_Radius_Direction': dp['dir_50'],
            '50kt_Wind_Longest_Radius_km': round(rad_50_l_km, 2) if rad_50_l_km is not None else None,
            '50kt_Wind_Shortest_Radius_km': round(rad_50_s_km, 2) if rad_50_s_km is not None else None,
            '30kt_Wind_Longest_Radius_Direction': dp['dir_30'],
            '30kt_Wind_Longest_Radius_km': round(rad_30_l_km, 2) if rad_30_l_km is not None else None,
            '30kt_Wind_Shortest_Radius_km': round(rad_30_s_km, 2) if rad_30_s_km is not None else None
        })
        
    df = pd.DataFrame(parsed_rows)
    
    if h['name'] == 'UNNAMED':
        fn = f"{h['year']}_UNNAMED_{h['tc_id']}.xlsx"
    else:
        fn = f"{h['year']}_{h['name']}.xlsx"
        
    full_fn = os.path.join(RESULTS_DIR, fn)
    df.to_excel(full_fn, index=False)
    
    closest_idx = enc['min_dist_idx']
    closest_dp = d_list[closest_idx]
    closest_dt = parse_time(closest_dp['time_str'])
    closest_wind_mps = closest_dp['max_wind'] * 0.51444 if closest_dp['max_wind'] is not None else None
    
    summary_list.append({
        'Typhoon_Name': f"{h['name']} ({h['tc_id']})",
        'Year_Month': closest_dt.strftime('%Y-%m'),
        'Closest_Distance_km': round(enc['min_dist'], 2),
        'Closest_Wind_Speed_mps': round(closest_wind_mps, 2) if closest_wind_mps is not None else None,
        'Closest_Center_Pressure_hPa': closest_dp['press']
    })

summary_df = pd.DataFrame(summary_list)
summary_df.to_excel(os.path.join(RESULTS_DIR, 'encounter_summary_list.xlsx'), index=False)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'encounter_summary_list.csv'), index=False)
print('Data processing complete!')

## 6. Yearly Track Plotting

In [ ]:
pin_path = r"target_pin.png"
if os.path.exists(pin_path):
    try:
        pin_img = plt.imread(pin_path)
    except Exception:
        pin_img = None
else:
    pin_img = None

encounters_by_year = {}
for enc in encounters:
    year = enc['header']['year']
    if year not in encounters_by_year:
        encounters_by_year[year] = []
    encounters_by_year[year].append(enc)

print("Generating yearly maps...")
for year, year_encs in sorted(encounters_by_year.items()):
    fig = plt.figure(figsize=(8, 8), dpi=100)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([TARGET_LON - 1.8, TARGET_LON + 1.8, TARGET_LAT - 1.7, TARGET_LAT + 1.7], crs=ccrs.PlateCarree())
    
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#f0eedb', edgecolor='#a1a1a1', linewidth=0.5)
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='#d6e6f2')
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), edgecolor='#787878', linewidth=0.8)
    ax.add_feature(cfeature.RIVERS.with_scale('50m'), edgecolor='#a2c4e8', linewidth=0.5)
    ax.add_feature(cfeature.LAKES.with_scale('50m'), facecolor='#d6e6f2', edgecolor='#787878', linewidth=0.5)
    ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False, linestyle=':', color='#cbd5e1')
    
    draw_target(ax, TARGET_LON, TARGET_LAT, THRESHOLD_KM, pin_img, 3.6)
    
    for enc in year_encs:
        name = enc['header']['name']
        tc_id = enc['header']['tc_id']
        d_list = enc['data']
        lons = [dp['lon'] for dp in d_list]
        lats = [dp['lat'] for dp in d_list]
        pressures = np.array([dp['press'] for dp in d_list])
        
        ax.plot(lons, lats, color='#93c5fd', linewidth=12.0, alpha=0.2, zorder=7)
        ax.plot(lons, lats, color='#60a5fa', linewidth=6.0, alpha=0.4, zorder=8)
        ax.plot(lons, lats, color='#1e3a8a', linewidth=2.5, alpha=0.9, label=f"{name} ({tc_id})", zorder=9)
        
        diff_press = np.maximum(0.0, BACKGROUND_PRESSURE - pressures)
        sizes = diff_press * 3.5
        ax.scatter(lons, lats, s=sizes, color='#2563eb', alpha=0.6, edgecolors='none', zorder=10)
        
    plt.title(f"JMA Typhoon Encounter Tracks ({year})", fontsize=14, fontweight='bold', color='#1e3a8a')
    plt.legend(loc='lower left', framealpha=0.9, fontsize=9)
    plt.savefig(os.path.join(RESULTS_DIR, f"{year}_encounter_paths.png"), bbox_inches='tight')
    plt.close()

## 7. Master Summary Map

In [ ]:
print("Generating master summary map...")
fig = plt.figure(figsize=(10, 10), dpi=120)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([MAP_MIN_LON, MAP_MAX_LON, MAP_MIN_LAT, MAP_MAX_LAT], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#f0eedb', edgecolor='#a1a1a1', linewidth=0.5)
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='#d6e6f2')
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), edgecolor='#787878', linewidth=0.8)
ax.add_feature(cfeature.RIVERS.with_scale('50m'), edgecolor='#a2c4e8', linewidth=0.5)
ax.add_feature(cfeature.LAKES.with_scale('50m'), facecolor='#d6e6f2', edgecolor='#787878', linewidth=0.5)
ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False, linestyle=':', color='#cbd5e1')

draw_target(ax, TARGET_LON, TARGET_LAT, THRESHOLD_KM, pin_img, 65.0)

for enc in encounters:
    d_list = enc['data']
    lons = [dp['lon'] for dp in d_list]
    lats = [dp['lat'] for dp in d_list]
    ax.plot(lons, lats, color='#2563eb', linewidth=0.6, alpha=0.15, zorder=6)

plt.title(f"Master Summary Map\nTotal Encounters: {len(encounters)}", fontsize=13, fontweight='bold', color='#1e3a8a')
plt.legend(loc='lower left', framealpha=0.9, fontsize=10)
plt.savefig(os.path.join(RESULTS_DIR, 'all_years_encounter_summary_map.png'), bbox_inches='tight')
plt.close()

## 8. Summary Encounter List

In [ ]:
pd.set_option('display.max_rows', 10)
summary_list_df = pd.read_excel(os.path.join(RESULTS_DIR, 'encounter_summary_list.xlsx'))
summary_list_df